# Taller evaluable: SQL Server y T-SQL
## Módulo 2 · Bloque de taller · Caso: biblioteca

Diplomado en Estrategias de Datos: Python, SQL Server y Big Data. Facultad de Estadística, Universidad Santo Tomás.

Este cuaderno se resuelve desde Python, conectado al motor SQL Server del Codespace. Antes de comenzar, el entorno de Python debe estar preparado y el kernel seleccionado, conforme al manual de consultas desde Python. Complete cada celda donde se indica.

## Parte práctica

### 0. Conexión y creación de la base de datos
Las dos celdas siguientes se conectan al motor, crean la base `biblioteca` y se conectan a ella. Ejecútelas tal cual.

In [1]:
import pymssql
import pandas as pd
import warnings
warnings.filterwarnings("ignore")   # oculta avisos no criticos de pandas

# Conexion a 'master' para crear la base. Si ya existe, se fuerza el cierre
# de otras conexiones antes de borrarla, para evitar el error 'base en uso'.
con_master = pymssql.connect(server="db", user="sa",
                             password="TuClave_Fuerte123",
                             database="master", autocommit=True)
cur = con_master.cursor()
cur.execute("""
IF DB_ID('biblioteca') IS NOT NULL
BEGIN
    ALTER DATABASE biblioteca SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
    DROP DATABASE biblioteca;
END
""")
cur.execute("CREATE DATABASE biblioteca")
con_master.close()
print("Base de datos biblioteca creada")

Base de datos biblioteca creada


In [2]:
# Conexion a la base biblioteca, que se usara en todo el taller
con = pymssql.connect(server="db", user="sa",
                      password="TuClave_Fuerte123",
                      database="biblioteca")
print("Conectado a biblioteca")

Conectado a biblioteca


### 1. Crear las tablas
Cree las tablas `lectores` y `prestamos` con esta especificación:

**lectores**: `lector_id` (entero, clave primaria), `nombre` (texto, obligatorio), `programa` (texto, admite nulo), `sede` (texto, obligatorio), `fecha_inscripcion` (fecha).

**prestamos**: `prestamo_id` (entero, clave primaria), `lector_id` (entero, clave foránea a `lectores`), `categoria` (texto), `titulo` (texto), `dias_prestamo` (entero), `multa` (decimal de 10 dígitos con 2 decimales), `fecha_prestamo` (fecha).

In [3]:
cur = con.cursor()

cur.execute("""
CREATE TABLE lectores (
    lector_id INT PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    programa VARCHAR(100) NULL,
    sede VARCHAR(100) NOT NULL,
    fecha_inscripcion DATE
)
""")

cur.execute("""
CREATE TABLE prestamos (
    prestamo_id INT PRIMARY KEY,
    lector_id INT FOREIGN KEY REFERENCES lectores(lector_id),
    categoria VARCHAR(50),
    titulo VARCHAR(100),
    dias_prestamo INT,
    multa DECIMAL(10,2),
    fecha_prestamo DATE
)
""")

con.commit()
print("Tablas creadas")

Tablas creadas


### 2. Cargar los datos
Ejecute esta celda para cargar los datos de ejemplo (sintéticos e ilustrativos).

In [5]:
# Datos sinteticos e ilustrativos (no son cifras reales)
lectores = [
    (1, 'Mariana Ospina', 'Medicina', 'Bogotá', '2023-02-19'),
    (2, 'Julián Cárdenas', 'Ingeniería', 'Chía', '2023-10-13'),
    (3, 'Natalia Bermúdez', 'Medicina', 'Chía', '2023-03-16'),
    (4, 'Sebastián Arango', None, 'Chía', '2025-07-28'),
    (5, 'Carolina Peña', 'Psicología', 'Chía', '2024-03-17'),
    (6, 'Esteban Gil', 'Economía', 'Villavicencio', '2023-07-17'),
    (7, 'Laura Montoya', None, 'Bogotá', '2023-02-10'),
    (8, 'Mateo Guerrero', 'Derecho', 'Chía', '2025-04-22'),
    (9, 'Isabela Cortés', 'Derecho', 'Chía', '2025-11-01'),
    (10, 'Samuel Rincón', 'Psicología', 'Chía', '2023-10-17'),
    (11, 'Gabriela León', 'Medicina', 'Bogotá', '2023-09-18'),
    (12, 'Nicolás Acosta', 'Psicología', 'Chía', '2023-10-03'),
    (13, 'Manuela Torres', 'Medicina', 'Tunja', '2024-10-27'),
    (14, 'David Salazar', 'Derecho', 'Chía', '2025-12-26'),
    (15, 'Antonia Vega', 'Ingeniería', 'Bogotá', '2025-08-04'),
    (16, 'Emilio Cárdenas', 'Medicina', 'Chía', '2023-12-25'),
    (17, 'Renata Ochoa', 'Derecho', 'Villavicencio', '2023-09-15'),
    (18, 'Simón Duarte', 'Psicología', 'Tunja', '2023-05-14'),
]

prestamos = [
    (1, 17, 'Tecnico', 'Calculo', 39, 0, '2026-06-21'),
    (2, 7, 'Ciencia', 'Cosmos', 11, 5400, '2026-06-15'),
    (3, 10, 'Historia', 'El Bogotazo', 5, 12500, '2026-05-15'),
    (4, 14, 'Novela', 'Delirio', 6, 3200, '2026-02-17'),
    (5, 7, 'Novela', 'Los ejercitos', 15, 1500, '2026-06-28'),
    (6, 7, 'Ciencia', 'El gen egoista', 41, 12500, '2026-06-25'),
    (7, 4, 'Tecnico', 'Bases de datos', 20, 0, '2026-01-23'),
    (8, 15, 'Tecnico', 'Algebra lineal', 6, 1500, '2026-05-13'),
    (9, 2, 'Tecnico', 'Calculo', 5, 0, '2026-04-17'),
    (10, 1, 'Novela', 'Los ejercitos', 24, 12500, '2026-01-11'),
    (11, 3, 'Novela', 'El olvido que seremos', 44, 1500, '2026-06-15'),
    (12, 8, 'Ciencia', 'Sapiens', 9, 21000, '2026-01-15'),
    (13, 15, 'Historia', 'El Bogotazo', 15, 0, '2026-01-26'),
    (14, 3, 'Historia', '1810', 23, 12500, '2026-01-14'),
    (15, 4, 'Historia', 'Bolivar', 28, 21000, '2026-03-22'),
    (16, 17, 'Novela', 'Los ejercitos', 9, 0, '2026-05-10'),
    (17, 6, 'Novela', 'La voragine', 44, 0, '2026-05-04'),
    (18, 4, 'Historia', 'El Bogotazo', 8, 8000, '2026-03-20'),
    (19, 7, 'Historia', 'Historia minima', 15, 5400, '2026-01-22'),
    (20, 1, 'Historia', 'Historia minima', 27, 0, '2026-04-07'),
    (21, 1, 'Ciencia', 'El gen egoista', 36, 5400, '2026-01-13'),
    (22, 17, 'Historia', 'La violencia en Colombia', 9, 21000, '2026-02-27'),
    (23, 10, 'Ciencia', 'El origen', 5, 21000, '2026-03-22'),
    (24, 3, 'Ciencia', 'Cosmos', 40, 3200, '2026-05-07'),
    (25, 5, 'Historia', 'La violencia en Colombia', 14, 0, '2026-01-13'),
    (26, 10, 'Ciencia', 'Sapiens', 6, 0, '2026-01-06'),
    (27, 8, 'Novela', 'Cien años de soledad', 3, 5400, '2026-04-16'),
    (28, 8, 'Historia', 'La violencia en Colombia', 28, 1500, '2026-02-07'),
    (29, 11, 'Ciencia', 'El gen egoista', 5, 0, '2026-06-13'),
    (30, 7, 'Tecnico', 'Algebra lineal', 19, 8000, '2026-03-22'),
    (31, 16, 'Tecnico', 'Calculo', 10, 8000, '2026-01-04'),
    (32, 10, 'Ciencia', 'El gen egoista', 23, 5400, '2026-03-22'),
    (33, 11, 'Novela', 'Cien años de soledad', 23, 0, '2026-01-19'),
    (34, 1, 'Ciencia', 'Sapiens', 18, 0, '2026-01-09'),
    (35, 18, 'Tecnico', 'Bases de datos', 13, 0, '2026-03-28'),
    (36, 14, 'Ciencia', 'Breve historia del tiempo', 44, 3200, '2026-02-13'),
    (37, 7, 'Historia', 'El Bogotazo', 36, 3200, '2026-06-05'),
    (38, 16, 'Ciencia', 'Breve historia del tiempo', 11, 0, '2026-03-06'),
    (39, 3, 'Tecnico', 'Calculo', 17, 0, '2026-01-12'),
    (40, 11, 'Novela', 'Los ejercitos', 5, 3200, '2026-01-06'),
    (41, 16, 'Historia', 'La violencia en Colombia', 42, 5400, '2026-06-25'),
    (42, 1, 'Novela', 'Cien años de soledad', 22, 21000, '2026-02-24'),
    (43, 17, 'Tecnico', 'Calculo', 17, 1500, '2026-06-11'),
    (44, 8, 'Tecnico', 'Redes', 23, 1500, '2026-03-20'),
    (45, 1, 'Tecnico', 'Estadistica aplicada', 4, 5400, '2026-06-04'),
    (46, 12, 'Novela', 'El olvido que seremos', 24, 1500, '2026-06-28'),
    (47, 15, 'Novela', 'El olvido que seremos', 39, 1500, '2026-01-15'),
    (48, 16, 'Tecnico', 'Redes', 32, 21000, '2026-04-17'),
]

cur = con.cursor()
cur.executemany("INSERT INTO lectores VALUES (%s, %s, %s, %s, %s)", lectores)
cur.executemany("INSERT INTO prestamos VALUES (%s, %s, %s, %s, %s, %s, %s)", prestamos)
con.commit()
print("Datos cargados:", len(lectores), "lectores y", len(prestamos), "prestamos")

Datos cargados: 18 lectores y 48 prestamos


## Consultas
Resuelva cada consulta en su celda. El resultado debe mostrarse como una tabla.

### Consulta 1. SELECT y TOP
Muestre los **primeros 8 lectores**, ordenados por `sede`.

In [6]:
# Consulta 1. SELECT y TOP
df = pd.read_sql("""
SELECT TOP (8) lector_id, nombre, sede, programa
FROM lectores
ORDER BY sede
""", con)
df


,lector_id,nombre,sede,programa
0,1,Mariana Ospina,Bogotá,Medicina
1,7,Laura Montoya,Bogotá,NaN
2,11,Gabriela León,Bogotá,Medicina
3,15,Antonia Vega,Bogotá,Ingeniería
4,5,Carolina Peña,Chía,Psicología
5,4,Sebastián Arango,Chía,NaN
6,3,Natalia Bermúdez,Chía,Medicina
7,2,Julián Cárdenas,Chía,Ingeniería


### Consulta 2. Variables y filtro
Defina en Python una variable `dias_minimos = 30` e insértela con una f-string. Muestre los préstamos cuyo `dias_prestamo` es **mayor o igual** al mínimo, ordenados de mayor a menor por días.

In [7]:
# Consulta 2. Variables y filtro
dias_minimos = 30

df = pd.read_sql(f"""
SELECT prestamo_id, titulo, dias_prestamo, multa
FROM prestamos
WHERE dias_prestamo >= {dias_minimos}
ORDER BY dias_prestamo DESC
""", con)
df

,prestamo_id,titulo,dias_prestamo,multa
0,11,El olvido que seremos,44,1500.0
1,17,La voragine,44,0.0
2,36,Breve historia del tiempo,44,3200.0
3,41,La violencia en Colombia,42,5400.0
4,6,El gen egoista,41,12500.0
5,24,Cosmos,40,3200.0
6,1,Calculo,39,0.0
7,47,El olvido que seremos,39,1500.0
8,37,El Bogotazo,36,3200.0
9,21,El gen egoista,36,5400.0


### Consulta 3. Limitar y paginar (TOP y OFFSET-FETCH)
Muestre los **4 préstamos con mayor multa** y, en otra tabla, los **4 siguientes** (paginación).

In [8]:
# Consulta 3. Limitar y paginar (TOP y OFFSET-FETCH)
top4 = pd.read_sql("""
SELECT TOP (4) prestamo_id, titulo, multa
FROM prestamos
ORDER BY multa DESC
""", con)

siguientes4 = pd.read_sql("""
SELECT prestamo_id, titulo, multa
FROM prestamos
ORDER BY multa DESC
OFFSET 4 ROWS FETCH NEXT 4 ROWS ONLY
""", con)

display(top4)
display(siguientes4)


,prestamo_id,titulo,multa
0,12,Sapiens,21000.0
1,15,Bolivar,21000.0
2,22,La violencia en Colombia,21000.0
3,23,El origen,21000.0


,prestamo_id,titulo,multa
0,42,Cien años de soledad,21000.0
1,48,Redes,21000.0
2,6,El gen egoista,12500.0
3,3,El Bogotazo,12500.0


### Consulta 4. Funciones de fecha
Muestre la fecha actual, la fecha dentro de **15 días** y los días transcurridos desde el **1 de marzo de 2026**. En otra tabla, el número de préstamos y el **promedio de días** por mes.

In [9]:
# Consulta 4. Funciones de fecha
fechas = pd.read_sql("""
SELECT GETDATE() AS fecha_actual,
       DATEADD(DAY, 15, GETDATE()) AS dentro_de_15_dias,
       DATEDIFF(DAY, '2026-03-01', GETDATE()) AS dias_desde_marzo
""", con)

por_mes = pd.read_sql("""
SELECT MONTH(fecha_prestamo) AS mes,
       COUNT(*) AS numero_prestamos,
       AVG(dias_prestamo * 1.0) AS promedio_dias
FROM prestamos
GROUP BY MONTH(fecha_prestamo)
ORDER BY mes
""", con)

display(fechas)
display(por_mes)

,fecha_actual,dentro_de_15_dias,dias_desde_marzo
0,2026-07-02 03:20:49.113,2026-07-17 03:20:49.113,123


,mes,numero_prestamos,promedio_dias
0,1,15,18.266666
1,2,5,21.800000
2,3,8,16.250000
3,4,4,16.750000
4,5,5,20.800000
5,6,11,25.272727


### Consulta 5. Funciones de texto
Para cada lector, muestre el nombre, el nombre en **minúsculas**, su longitud y el **apellido** (la parte que va después del primer espacio).

In [10]:
# Consulta 5. Funciones de texto
df = pd.read_sql("""
SELECT nombre,
       LOWER(nombre) AS nombre_minusculas,
       LEN(nombre) AS longitud,
       SUBSTRING(nombre, CHARINDEX(' ', nombre) + 1, LEN(nombre)) AS apellido
FROM lectores
""", con)
df

,nombre,nombre_minusculas,longitud,apellido
0,Mariana Ospina,mariana ospina,14,Ospina
1,Julián Cárdenas,julián cárdenas,15,Cárdenas
2,Natalia Bermúdez,natalia bermúdez,16,Bermúdez
3,Sebastián Arango,sebastián arango,16,Arango
4,Carolina Peña,carolina peña,13,Peña
5,Esteban Gil,esteban gil,11,Gil
6,Laura Montoya,laura montoya,13,Montoya
7,Mateo Guerrero,mateo guerrero,14,Guerrero
8,Isabela Cortés,isabela cortés,14,Cortés
9,Samuel Rincón,samuel rincón,13,Rincón


### Consulta 6. Conversión de tipos
Muestre la `multa` convertida a entero y la `fecha_prestamo` como texto en formato **dd.mm.aaaa** (estilo 104), para los 5 primeros préstamos. En otra tabla, demuestre `TRY_CONVERT` con '2024' y con 'x12'.

In [11]:
# Consulta 6. Conversión de tipos
conversiones = pd.read_sql("""
SELECT TOP (5) prestamo_id,
       CAST(multa AS INT) AS multa_entera,
       CONVERT(VARCHAR, fecha_prestamo, 104) AS fecha_texto
FROM prestamos
""", con)

pruebas = pd.read_sql("""
SELECT TRY_CONVERT(INT, '2024') AS convierte_2024,
       TRY_CONVERT(INT, 'x12') AS convierte_x12
""", con)

display(conversiones)
display(pruebas)

,prestamo_id,multa_entera,fecha_texto
0,1,0,21.06.2026
1,2,5400,15.06.2026
2,3,12500,15.05.2026
3,4,3200,17.02.2026
4,5,1500,28.06.2026


,convierte_2024,convierte_x12
0,2024,None


### Consulta 7. Manejo de nulos
Muestre el nombre, el `programa` original y el programa corregido, reemplazando los nulos por **'No registrado'** con `COALESCE`.

In [12]:
# Consulta 7. Manejo de nulos
df = pd.read_sql("""
SELECT nombre,
       programa AS programa_original,
       COALESCE(programa, 'No registrado') AS programa_corregido
FROM lectores
""", con)
df


,nombre,programa_original,programa_corregido
0,Mariana Ospina,Medicina,Medicina
1,Julián Cárdenas,Ingeniería,Ingeniería
2,Natalia Bermúdez,Medicina,Medicina
3,Sebastián Arango,NaN,No registrado
4,Carolina Peña,Psicología,Psicología
5,Esteban Gil,Economía,Economía
6,Laura Montoya,NaN,No registrado
7,Mateo Guerrero,Derecho,Derecho
8,Isabela Cortés,Derecho,Derecho
9,Samuel Rincón,Psicología,Psicología


### Consulta 8. Lógica condicional (CASE e IIF)
Clasifique cada préstamo con `IIF` en 'Con multa' o 'Sin multa' según la multa sea mayor que cero, y con `CASE` la duración en 'Largo' (30 días o más), 'Medio' (15 a 29) o 'Corto' (menos de 15).

In [13]:
df = pd.read_sql("""
SELECT prestamo_id, titulo, multa, dias_prestamo,
       IIF(multa > 0, 'Con multa', 'Sin multa') AS estado_multa,
       CASE
           WHEN dias_prestamo >= 30 THEN 'Largo'
           WHEN dias_prestamo >= 15 THEN 'Medio'
           ELSE 'Corto'
       END AS duracion
FROM prestamos
""", con)
df

,prestamo_id,titulo,multa,dias_prestamo,estado_multa,duracion
0,1,Calculo,0.0,39,Sin multa,Largo
1,2,Cosmos,5400.0,11,Con multa,Corto
2,3,El Bogotazo,12500.0,5,Con multa,Corto
3,4,Delirio,3200.0,6,Con multa,Corto
4,5,Los ejercitos,1500.0,15,Con multa,Medio
5,6,El gen egoista,12500.0,41,Con multa,Largo
6,7,Bases de datos,0.0,20,Sin multa,Medio
7,8,Algebra lineal,1500.0,6,Con multa,Corto
8,9,Calculo,0.0,5,Sin multa,Corto
9,10,Los ejercitos,12500.0,24,Con multa,Medio


### Consulta 9. Expresiones de tabla comunes (CTE)
Con una CTE, calcule por categoría el **total de multa**, el número de préstamos y el **promedio de días**. Muestre solo las categorías cuyo total de multa supera **30000**.

In [14]:
# Consulta 9. Expresiones de tabla comunes (CTE)
df = pd.read_sql("""
WITH resumen AS (
    SELECT categoria,
           SUM(multa) AS total_multa,
           COUNT(*) AS numero_prestamos,
           AVG(dias_prestamo * 1.0) AS promedio_dias
    FROM prestamos
    GROUP BY categoria
)
SELECT categoria, total_multa, numero_prestamos, promedio_dias
FROM resumen
WHERE total_multa > 30000
ORDER BY total_multa DESC
""", con)
df



,categoria,total_multa,numero_prestamos,promedio_dias
0,Historia,90500.0,12,20.833333
1,Ciencia,77100.0,12,20.750000
2,Novela,51300.0,12,21.500000
3,Tecnico,46900.0,12,17.083333


### Consulta 10. Funciones de ventana (OVER)
Con `ROW_NUMBER` y `PARTITION BY`, obtenga los **2 préstamos con mayor multa de cada categoría**.

In [15]:
# Consulta 10. Funciones de ventana (OVER)
df = pd.read_sql("""
WITH ranking AS (
    SELECT categoria, titulo, multa,
           ROW_NUMBER() OVER (PARTITION BY categoria ORDER BY multa DESC) AS posicion
    FROM prestamos
)
SELECT categoria, titulo, multa, posicion
FROM ranking
WHERE posicion <= 2
ORDER BY categoria, posicion
""", con)
df

,categoria,titulo,multa,posicion
0,Ciencia,Sapiens,21000.0,1
1,Ciencia,El origen,21000.0,2
2,Historia,La violencia en Colombia,21000.0,1
3,Historia,Bolivar,21000.0,2
4,Novela,Cien años de soledad,21000.0,1
5,Novela,Los ejercitos,12500.0,2
6,Tecnico,Redes,21000.0,1
7,Tecnico,Algebra lineal,8000.0,2


### Cierre
Al terminar, cierre la conexión con el motor.

In [16]:
con.close()
print("Conexión cerrada")

Conexión cerrada
